# Labels by Source

Where the annotated texts come from, and whether the labels vary by source.
Uses `folder` rather than `dolma_source`, because folder separates the Common
Crawl subsets (head / middle / tail, web vs. news) that `source` collapses.

All labels here are the adjudicator's.

In [ ]:
import pandas as pd
import matplotlib.ticker as mticker

from nb_utils import setup_plots, load, short_name, DIMENSIONS, SCALE_15

plt = setup_plots()

ANNOTATOR = 'tejo9855'

FOLDER_PALETTE = {
    'cc_en_head':                 '#1f77b4',
    'cc_en_middle':               '#4da3e0',
    'cc_en_tail':                 '#aec7e8',
    'cc_news_head':               '#d62728',
    'cc_news_middle':             '#e87070',
    'cc_news_tail':               '#f2b4b4',
    'falcon-refinedweb-filtered': '#ff7f0e',
    'reddit':                     '#2ca02c',
    'c4-filtered':                '#9467bd',
    'books':                      '#8c564b',
    'wiki':                       '#e377c2',
    'wikiref_megawika':           '#bcbd22',
}
folder_colors = lambda index: [FOLDER_PALETTE.get(f, '#aec7e8') for f in index]

task_frames = {
    'Setting':        load('setting'),
    'Agency':         load('agency'),
    'Event relation': load('event_relation'),
}
for task, df in task_frames.items():
    print(f'{task:16s} {len(df)} instances, {df["folder"].nunique()} folders')

## 1. Folder distribution per task

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

for ax, (task, df) in zip(axes, task_frames.items()):
    counts = df['folder'].value_counts().sort_values()
    bars = ax.barh(counts.index, counts.to_numpy(),
                   color=folder_colors(counts.index), edgecolor='white')
    for bar, val in zip(bars, counts.to_numpy()):
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                str(val), va='center', fontsize=9)
    ax.set_title(f'{task} (n={len(df)})', fontsize=11)
    ax.set_xlabel('Instances')
    ax.set_xlim(0, counts.max() * 1.15)

plt.suptitle(f'Folder distribution of annotated texts — {ANNOTATOR}', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Side by side across tasks

In [ ]:
combined = pd.DataFrame({task: df['folder'].value_counts()
                         for task, df in task_frames.items()}).fillna(0).astype(int)
combined = combined.loc[combined.sum(axis=1).sort_values(ascending=False).index]

ax = combined.plot(kind='bar', figsize=(12, 5), edgecolor='white',
                   color=['#1f77b4', '#ff7f0e', '#2ca02c'])
ax.set_title(f'Folder counts by task — {ANNOTATOR}', fontsize=13)
ax.set_xlabel('Folder')
ax.set_ylabel('Instances')
ax.tick_params(axis='x', rotation=30)
ax.legend(title='Task')
plt.tight_layout()
plt.show()

print(combined.to_string())

## 3. Folder proportions

In [ ]:
proportions = combined.div(combined.sum(axis=0), axis=1) * 100

ax = proportions.T.plot(kind='bar', stacked=True, figsize=(8, 5), edgecolor='white',
                        color=folder_colors(proportions.index))
ax.set_title(f'Folder proportions by task — {ANNOTATOR}', fontsize=13)
ax.set_xlabel('Task')
ax.set_ylabel('Share of task')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Folder', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

print(proportions.round(1).to_string())

## 4. Likert ratings by folder

Mean rating per dimension, by source folder. Both tasks are 1–5, so the axis is
fixed to the full scale and the folders are directly comparable across panels.

Folders contributing only a handful of instances produce unstable means, so the
instance count is printed beside each bar.

In [ ]:
def ratings_by_folder(task, title):
    df   = load(task)
    dims = DIMENSIONS[task]
    cols = [f'{d}_{ANNOTATOR}' for d in dims if f'{d}_{ANNOTATOR}' in df.columns]
    sizes = df.groupby('folder').size()

    # One folder order for every panel. Sorting each panel independently under
    # sharey would put the bars in one order and the tick labels in another.
    order = df.groupby('folder')[cols].mean().mean(axis=1).sort_values().index

    fig, axes = plt.subplots(1, len(cols), figsize=(3.6 * len(cols), 6), sharey=True)
    for ax, (dim, col) in zip(axes, zip(dims, cols)):
        means = df.groupby('folder')[col].mean().reindex(order)
        ax.barh(list(means.index), means.to_numpy(),
                color=folder_colors(means.index), edgecolor='white')
        for i, (folder, val) in enumerate(means.items()):
            ax.text(val + 0.05, i, f'{val:.2f}  (n={sizes[folder]})',
                    va='center', fontsize=8)
        ax.set_title(short_name(dim), fontsize=10)
        ax.set_xlabel('Mean rating')
        ax.set_xlim(SCALE_15[0] - 0.1, SCALE_15[-1] + 0.9)
        ax.set_xticks(SCALE_15)

    plt.suptitle(f'{title} — {ANNOTATOR}', fontsize=13)
    plt.tight_layout()
    plt.show()


ratings_by_folder('setting', 'Mean setting ratings by folder')
ratings_by_folder('agency',  'Mean agency ratings by folder')

## 5. Event relation by folder

`span1_is_event` / `span2_is_event` are genuinely boolean, so they are shown as a
rate. `temporal_order` and `causality_rating` are categorical, so they are shown
as the composition of each folder rather than a mean.

In [ ]:
event_df = load('event_relation')
bool_cols = [f'{d}_{ANNOTATOR}' for d in ('span1_is_event', 'span2_is_event')
             if f'{d}_{ANNOTATOR}' in event_df.columns]
cat_cols  = [f'{d}_{ANNOTATOR}' for d in ('temporal_order', 'causality_rating')
             if f'{d}_{ANNOTATOR}' in event_df.columns]
sizes = event_df.groupby('folder').size()

# Shared folder order across panels, as above.
event_order = (event_df.groupby('folder')[bool_cols]
               .apply(lambda g: g.astype('boolean').mean().mean())
               .sort_values().index)

fig, axes = plt.subplots(1, len(bool_cols), figsize=(6 * len(bool_cols), 6), sharey=True)
for ax, col in zip(axes, bool_cols):
    rates = (event_df.groupby('folder')[col]
             .apply(lambda x: x.astype('boolean').mean() * 100).reindex(event_order))
    ax.barh(list(rates.index), rates.to_numpy(),
            color=folder_colors(rates.index), edgecolor='white')
    for i, (folder, val) in enumerate(rates.items()):
        ax.text(val + 1, i, f'{val:.0f}%  (n={sizes[folder]})', va='center', fontsize=8)
    ax.set_title(short_name(col.replace(f'_{ANNOTATOR}', '')), fontsize=10)
    ax.set_xlabel('% judged an event')
    ax.set_xlim(0, 128)

plt.suptitle(f'Span-is-event rates by folder — {ANNOTATOR}', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical dimensions: what each folder's answers are made of.
fig, axes = plt.subplots(1, len(cat_cols), figsize=(8 * len(cat_cols), 6))
for ax, col in zip(axes, cat_cols):
    comp = (event_df.groupby('folder')[col].value_counts(normalize=True)
            .unstack(fill_value=0) * 100)
    comp = comp.loc[comp.index.sort_values()]
    comp.plot(kind='barh', stacked=True, ax=ax, edgecolor='white', width=0.75,
              colormap='tab20')
    ax.set_title(short_name(col.replace(f'_{ANNOTATOR}', '')), fontsize=10)
    ax.set_xlabel('Share of answers')
    ax.set_ylabel('')
    ax.xaxis.set_major_formatter(mticker.PercentFormatter())
    ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')

plt.suptitle(f'Event-relation answer composition by folder — {ANNOTATOR}', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Summary table

In [ ]:
pivot = pd.DataFrame({task: df['folder'].value_counts()
                      for task, df in task_frames.items()}).fillna(0).astype(int)
pivot['Total'] = pivot.sum(axis=1)
print(pivot.sort_values('Total', ascending=False).to_string())